In [1]:
%pip install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 26.5 MB/s  0:00:006m0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 69.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 77.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.8/32.8 MB 82.0 MB/s  0:00:00m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 95.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.3/16.3 MB 75.0 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.9/835.9 kB 70.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 89.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 40.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 89.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 91.3 MB/s  0:00:00
   ━━━━━

In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import  LatentDirichletAllocation, NMF
import spacy
import pyLDAvis
import pyLDAvis.lda_model
from sklearn.feature_extraction.text import TfidfVectorizer
import matplotlib.pyplot as plt
import textwrap
import numpy as np
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel

/opt/python/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-06 16:45:24.866993: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-06 16:45:24.975379: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-06 16:45:35.567617: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different 

In [5]:
df = pd.read_csv("archelect_search.csv",sep=",")
df = df[df["contexte-election"]=="législatives"]
print(df.shape)

/tmp/ipykernel_1818/913678588.py:1: DtypeWarning: Columns (0: departement-nom, 1: departement-insee, 2: identifiant de circonscription, 3: pdf, 4: suppleant-nom, 5: suppleant-prenom, 6: suppleant-sexe, 7: suppleant-age, 8: suppleant-age-calcule, 9: suppleant-age-tranche, 10: suppleant-profession, 11: suppleant-mandat-en-cours, 12: suppleant-mandat-passe, 13: suppleant-associations, 14: suppleant-autres-statuts, 15: suppleant-soutien, 16: suppleant-liste, 17: suppleant-decorations) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("archelect_search.csv",sep=",")


(32625, 42)


In [6]:
import os
import pandas as pd

TEXT_DIR = "1981/legislatives"

def load_texts_from_dir(text_dir):
    texts = []
    meta = []

    for filename in os.listdir(text_dir):
        if not filename.endswith(".txt"):
            continue
        
        path = os.path.join(text_dir, filename)
        doc_id = filename.replace(".txt", "")
        
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            text = f.read().strip()  # enlever les espaces en début/fin
            if len(text) == 0:
                continue  # ignorer les fichiers vides
        
        texts.append(text)
        meta.append({
            "id": doc_id,
            "text": text
        })

    df = pd.DataFrame(meta)
    print(f"Nombre de documents chargés : {len(df)}")
    return df

# Charger tous les textes
df = load_texts_from_dir(TEXT_DIR)

# Afficher les premières lignes pour vérification
df.head()

Nombre de documents chargés : 3182


,id,text
0,EL135_L_1981_06_062_01_1_PF_02,Sciences Po / fonds CEVIPOF\nElection Législat...
1,EL136_L_1981_06_069_01_1_PF_07,Sciences Po / fonds CEVIPOF\nREPUBLIQUE FRANÇA...
2,EL134_L_1981_06_002_01_1_PF_01,ÉLECTIONS LÉGISLATIVES - JUIN 1981\n1re Circon...
3,EL135_L_1981_06_042_02_2_PF_01,2me Circonscription de Saint-Etienne - ELECTIO...
4,EL135_L_1981_06_060_05_1_PF_01,Élections législatives de juin 1981 5e circons...


In [7]:
STOPWORDS = [x.strip() for x in open('data/stop_word_fr.txt').readlines()]
nlp = spacy.load("fr_core_news_sm", disable=["parser"])
df['lemmatized_text'] = [" ".join([token.lemma_ for token in doc]) for doc in nlp.pipe(df['text'])]


,index,id,text,lemmatized_text
0,0,EL135_L_1981_06_062_01_1_PF_02,Sciences Po / fonds CEVIPOF\nElection Législat...,science po / fonds cevipof \n election législa...
1,1,EL136_L_1981_06_069_01_1_PF_07,Sciences Po / fonds CEVIPOF\nREPUBLIQUE FRANÇA...,science po / fonds cevipof \n republique franç...
2,2,EL134_L_1981_06_002_01_1_PF_01,ÉLECTIONS LÉGISLATIVES - JUIN 1981\n1re Circon...,élection législatif - juin 1981 \n 1re circons...
3,3,EL135_L_1981_06_042_02_2_PF_01,2me Circonscription de Saint-Etienne - ELECTIO...,2me circonscription de Saint-Etienne - ELECTIO...
4,4,EL135_L_1981_06_060_05_1_PF_01,Élections législatives de juin 1981 5e circons...,élection législatif de juin 1981 5e circonscri...
...,...,...,...,...
3177,3177,EL134_L_1981_06_010_02_2_PF_02,Jean Weinling\nMAIRE DE BAR-SUR-SEINE CANDIDAT...,Jean Weinling \n maire de bar-sur-seine candid...
3178,3178,EL135_L_1981_06_056_06_1_PF_05,Sciences Po / fonds CEVIPOF\nELECTIONS LEGISLA...,science po / fonds cevipof \n election legisla...
3179,3179,EL134_L_1981_06_025_03_1_PF_02,ELECTIONS LEGISLATIVES - JUIN 1981 - 3ème CIRC...,election legislatives - juin 1981 - 3èm circon...
3180,3180,EL135_L_1981_06_037_04_1_PF_05,4. CIRCONSCRIPTION\nDépartement d'Indre-et-loi...,4 . circonscription \n département de indre-et...


In [10]:
df.to_csv("df.txt",sep=";")

In [11]:
df = pd.read_csv("df.txt",sep=";")

In [15]:
df_vectorizer = CountVectorizer(
    min_df=0.005,
    max_df=0.7,
    ngram_range=(1, 2)
)

X = df_vectorizer.fit_transform(df["lemmatized_text"])

In [ ]:
perplexities = []
for k in range(5, 16, 1):
    lda_test = LatentDirichletAllocation(n_components=k, random_state=42, learning_method='batch')
    lda_test.fit(X)
    perplexities.append(lda_test.perplexity(X))
    print(f'Nombre de topics testés: {k} / Perplexité: {lda_test.perplexity(X):.2f}')

In [ ]:
lda = LatentDirichletAllocation(
    n_components=10,    # nombre de topics
    random_state=42,
    learning_method='batch'  # 'batch' pour un dataset modéré
)

# Entraîne le modèle sur la matrice document-terme
lda.fit(X)